# Mixed Preference BO：複数ペアの逐次最適化

連続変数とカテゴリ変数を含む選好学習です。1ラウンドに複数の独立した2候補比較を行います。比較の順序は `[winner, loser]` です。

In [ ]:
import torch

from robotorchan.optim.preference_loop import run_mixed_preference_batch_loop

torch.manual_seed(0)
dtype = torch.double

## 1. 初期比較データ

第1列は連続値、第2列はカテゴリ（0または1）です。

In [ ]:
datapoints = torch.tensor(
    [[0.0, 0.0], [0.3, 1.0], [0.7, 0.0], [1.0, 1.0]], dtype=dtype
)
comparisons = torch.tensor([[1, 0], [2, 3], [1, 2]], dtype=torch.long)
bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]], dtype=dtype)
fixed_features_list = [{1: 0.0}, {1: 1.0}]

## 2. 比較フィードバック

実際の運用では人間・実験から各ペアの勝者を受け取ります。ここでは再現可能な疑似utilityを使います。

In [ ]:
def feedback(pairs: torch.Tensor) -> list[int]:
    utility = torch.sin(2 * torch.pi * pairs[..., 0]) + 0.2 * pairs[..., 1]
    return utility.argmax(dim=1).tolist()

## 3. 逐次最適化

各ラウンドで1回モデルを学習し、独立したq=2ペアを2組提案します。重複時にはランダム候補からEUBOの高い代替ペアを選びます。

In [ ]:
updated_points, updated_comparisons, batches = run_mixed_preference_batch_loop(
    datapoints,
    comparisons,
    cat_dims=[1],
    bounds=bounds,
    fixed_features_list=fixed_features_list,
    feedback=feedback,
    pairs_per_step=2,
    steps=2,
    min_pair_distance=0.05,
    diversity_samples=64,
    fit_optimizer_kwargs={"options": {"maxiter": 10}},
    num_restarts=2,
    raw_samples=8,
    options={"maxiter": 10},
)
print("datapoints:", updated_points.shape)
print("comparisons:", updated_comparisons.shape)
for step, batch in enumerate(batches, start=1):
    print(f"round {step}:", batch)

## 4. 注意事項

- 各ペアはq=2のAnalytic EUBOで評価します。q>2のjoint EUBOやlistwise順位学習ではありません。
- 同一ラウンドの重複を回避しますが、既存観測点との重複回避は保証しません。
- カテゴリの組合せが多い場合、列挙コストが増加します。
- 実データで利用する場合は`feedback`を評価・実験結果の取得処理に置き換えます。